# What is PAIDF AnomalyGen?

PAIDF AnomalyGen is a diffusion-based pipeline for synthetic anomaly data generation in a few-shot scenario.

In this series of tutorials, we will walk you through the following steps:

0. Setting up the environment
1. Training the PAIDF AnomalyGen modules
2. (Optional) Automatic mask placement
3. Generating synthetic anomaly data
4. Pseudo-labeling on generated data

## Important Notes

### Syntax Differences between Notebook and Terminal

The code in this notebook is designed to run within a **Jupyter notebook**. If you want to run it in a bash terminal, ensure you use the correct syntax for the command line.

Syntax differences:

* Environment variables:
  * Notebook: `{LOCAL_PROJECT_DIR}`
  * Bash: `${LOCAL_PROJECT_DIR}`
  * Notebook: `{HF_TOKEN}`
  * Bash: `${HF_TOKEN}`
* Escape (`\`) in the command:
  * Notebook: `conda run -n cosmos-predict2 bash -c "CUDA_HOME=\$CONDA_PREFIX" pip install ...`
  * Bash: `conda activate cosmos-predict2 && CUDA_HOME=$CONDA_PREFIX pip install ...`

We have provided the equivalent bash commands in some cells which take a long time to run. You can copy and paste them into your terminal if you prefer to run them outside of the notebook environment. These cells have a collapsed section `The equivalent command in the bash terminal. (Click to show)` that you can click to expand.

### Live Stream Logging with `conda run`

When executing commands using `conda run` in the notebook, you can append `--live-stream` flag to see the live logs in the notebook output.

Example:

`conda run --live-stream -n cosmos-predict2 pip install -r "requirements-conda-cuda128.txt"`

## 0. Setting Up the Environment

This notebook was tested with the following environment:

* Operating System: Ubuntu 24.04 (x86_64, UEFI)
* Notebook Kernel: Python 3.12
* GPU: 1× RTX PRO 6000 Blackwell
* NVIDIA Driver: 580
* Hugging Face Account: A valid Hugging Face token is required to download Cosmos-Predict2 models.


### 0.1 Setting Up the Environment

This notebook requires the user to set the environment variable `LOCAL_PROJECT_DIR` to the path of the PAIDF AnomalyGen repo. Remember to replace `FIXME` placeholder below with the correct path.

Also it's optional to increase `MAX_JOBS` and `NVCC_THREADS` if your memory size permits.

**Important: This step might take a while to complete (> 1 hour) depending on your machine and network speed. Please be patient.**

<details>
<summary> <b> The equivalent setup script in the bash terminal. (Click to show) <b> </summary>

```bash
LOCAL_PROJECT_DIR="FIXME"
cd ${LOCAL_PROJECT_DIR}

curl https://repo.anaconda.com/archive/Anaconda3-2022.05-Linux-x86_64.sh -o Anaconda3-2022.05-Linux-x86_64.sh
./Anaconda3-2022.05-Linux-x86_64.sh -b -u -p ${LOCAL_PROJECT_DIR}/anaconda3

conda init bash
conda env create --file cosmos-predict2-cuda128.yaml
conda activate cosmos-predict2

# Ensure the cosmos-predict2 environment is activated.
# Note that installing `flash-attn` might require a great amount of time to compile. Please be patient.
pip install -r "requirements-conda-cuda128.txt"
MAX_JOBS=4 NVCC_THREADS=2 pip install --no-build-isolation flash-attn==2.8.3

# Fix the Transformer engine linking issues in the conda environment.
ln -sf $CONDA_PREFIX/lib/python3.12/site-packages/nvidia/*/include/* $CONDA_PREFIX/include/
ln -sf $CONDA_PREFIX/lib/python3.12/site-packages/nvidia/*/include/* $CONDA_PREFIX/include/python3.12

# Install Transformer engine.
pip install --no-build-isolation transformer-engine[pytorch]==2.13.0

# Install Apex.
# Note that installing `apex` might require a great amount of time to compile. Please be patient.
unset TORCH_CUDA_ARCH_LIST
MAX_JOBS=4 NVCC_THREADS=2 CUDA_HOME=$CONDA_PREFIX pip install -v --disable-pip-version-check --no-cache-dir --no-build-isolation --config-settings "--build-option=--cpp_ext --cuda_ext" git+https://github.com/NVIDIA/apex.git

# Install p7zip to extract the .rar file.
conda install -c conda-forge p7zip -y

# Install Huggingface Hub to download the pretrained models.
pip install huggingface_hub
```

</details>

In [ ]:
# Set `LOCAL_PROJECT_DIR` for PAIDF AnomalyGen.
LOCAL_PROJECT_DIR="FIXME"
# Set the working directory to this path for the shell.
%cd {LOCAL_PROJECT_DIR}
# Use `cd ${LOCAL_PROJECT_DIR}` if you are copy-pasting this into a terminal.

# (Optional) Install conda.
# If you already have conda installed, you can comment out the following 2 steps.
!curl https://repo.anaconda.com/archive/Anaconda3-2022.05-Linux-x86_64.sh -o Anaconda3-2022.05-Linux-x86_64.sh
!bash Anaconda3-2022.05-Linux-x86_64.sh -b -u -p {LOCAL_PROJECT_DIR}/anaconda3

# Add conda in the `PATH`.
import os
os.environ["PATH"] = f"{LOCAL_PROJECT_DIR}/anaconda3/bin:" + os.environ["PATH"]

# Create the environment for PAIDF AnomalyGen.
!conda init bash 
!conda env create --file cosmos-predict2-cuda128.yaml

# Install the requirements.
!conda run -n cosmos-predict2 pip install -r "requirements-conda-cuda128.txt"
!conda run -n cosmos-predict2 bash -c "MAX_JOBS=4 NVCC_THREADS=2 pip install --no-build-isolation flash-attn==2.8.3"

# Fix the Transformer engine linking issues in the conda environment.
!conda run -n cosmos-predict2 bash -c 'ln -sf $CONDA_PREFIX/lib/python3.12/site-packages/nvidia/*/include/* $CONDA_PREFIX/include/'
!conda run -n cosmos-predict2 bash -c 'ln -sf $CONDA_PREFIX/lib/python3.12/site-packages/nvidia/*/include/* $CONDA_PREFIX/include/python3.12'

# Install Transformer engine.
!conda run -n cosmos-predict2 pip install --no-build-isolation transformer-engine[pytorch]==2.13.0

# Install Apex.
# Note that installing `apex` might require a great amount of time to compile. Please be patient.
!conda run -n cosmos-predict2 bash -c "unset TORCH_CUDA_ARCH_LIST; MAX_JOBS=4 NVCC_THREADS=2 CUDA_HOME=\$CONDA_PREFIX pip install -v --disable-pip-version-check --no-cache-dir --no-build-isolation --config-settings \"--build-option=--cpp_ext --cuda_ext\" git+https://github.com/NVIDIA/apex.git"

# Install p7zip to extract the .rar file.
!conda run -n cosmos-predict2 bash -c "conda install -c conda-forge p7zip -y"

# Install Huggingface Hub to download the pretrained models.
!conda run -n cosmos-predict2 pip install huggingface_hub

Ensure the nvrtc related libraries are available.

You should see something like this:

```bash
libnvrtc.so.12 (libc6,x86-64) => /lib/x86_64-linux-gnu/libnvrtc.so.12
...
```

In [ ]:
!ldconfig -p | grep libnvrtc*

If you ran into errors for missing these libraries, please try the following workaround to resolve the issues.

In [ ]:
# (Optional) Update the linker.
!bash -c "mkdir '$HOME/.cosmos_lib'"
!conda run -n cosmos-predict2 bash -c 'ln -s $CONDA_PREFIX/targets/x86_64-linux/lib/libnvrtc-builtins.so.12.8.93 $HOME/.cosmos_lib/libnvrtc-builtins.so'
!conda run -n cosmos-predict2 bash -c 'ln -s $CONDA_PREFIX/targets/x86_64-linux/lib/libnvrtc-builtins.so.12.8.93 $HOME/.cosmos_lib/libnvrtc-builtins.so.12.8'
!conda run -n cosmos-predict2 bash -c 'ln -s $CONDA_PREFIX/targets/x86_64-linux/lib/libnvrtc-builtins.so.12.8.93 $HOME/.cosmos_lib/libnvrtc-builtins.so.12.8.93'
!conda run -n cosmos-predict2 bash -c 'ln -s $CONDA_PREFIX/targets/x86_64-linux/lib/libnvrtc.so.12.8.93 $HOME/.cosmos_lib/libnvrtc.so'
!conda run -n cosmos-predict2 bash -c 'ln -s $CONDA_PREFIX/targets/x86_64-linux/lib/libnvrtc.so.12.8.93 $HOME/.cosmos_lib/libnvrtc.so.12'
!conda run -n cosmos-predict2 bash -c 'ln -s $CONDA_PREFIX/targets/x86_64-linux/lib/libnvrtc.so.12.8.93 $HOME/.cosmos_lib/libnvrtc.so.12.8.93'

!sudo bash -c "echo '$HOME/.cosmos_lib' > /etc/ld.so.conf.d/conda-env.conf"
!sudo ldconfig


### 0.2 Preparing the Dataset

#### 0.2.1 Preparing the MeiweiPCB Dataset

We will use the [MeiweiPCB](https://gitee.com/youtang1993/meiwei-pcb-surface-defect-dataset) dataset demonstrate training PAIDF AnomalyGen. The dataset consists of 2 classes (1 normal class and 1 defect classes), along with their corresponding defect masks.

The directory hierarchy is organized as follows:

```shell
data/MeiweiPCB
├── Anomaly_test
│   └── normal_img/         # Normal images used for generation
└── train_downsample
    └── PCB
        ├── anomaly_image/  # Defect images for training
        │   └── defect/
        └── mask/           # Corresponding masks for the defect type
            └── defect/
```



In [ ]:
# Download and preprocess the MeiweiPCB dataset.
!mkdir -p datasets/meiweipcb_download

# Download the zip files
!curl -L https://gitee.com/youtang1993/meiwei-pcb-surface-defect-dataset/raw/master/MeiweiPCB/images.zip -o datasets/meiweipcb_download/images.zip
!curl -L https://gitee.com/youtang1993/meiwei-pcb-surface-defect-dataset/raw/master/MeiweiPCB/images_nor.zip -o datasets/meiweipcb_download/images_nor.zip
!curl -L https://gitee.com/youtang1993/meiwei-pcb-surface-defect-dataset/raw/master/MeiweiPCB/mask.zip -o datasets/meiweipcb_download/mask.zip

# Unzip the files
!conda run -n cosmos-predict2 bash -c "unzip -o datasets/meiweipcb_download/images.zip -d datasets/meiweipcb_download/"
!conda run -n cosmos-predict2 bash -c "unzip -o datasets/meiweipcb_download/images_nor.zip -d datasets/meiweipcb_download/"
!conda run -n cosmos-predict2 bash -c "unzip -o datasets/meiweipcb_download/mask.zip -d datasets/meiweipcb_download/"

# Convert to the required format
!conda run -n cosmos-predict2 bash -c "python tutorial/meiweipcb_data_convert.py --source datasets/meiweipcb_download --target datasets/MeiweiPCB"

# Remove the downloaded zip files to free up disk space
!rm -r datasets/meiweipcb_download

In [ ]:
# Show the dataset structure.
!find datasets/MeiweiPCB/train_downsample | sed -e "s/[^-][^\/]*\// |/g" -e "s/|\([^ ]\)/|-\1/"

#### 0.2.2 Preparing Custom Dataset

* Before model training, you should categorize your anomaly dataset into several classes with "Texture" and "Anomaly type" information. We suggest user to classify this via domain knowhow (The step is **important** since it groups data with similar characteristics together. We suggest user to categorize your data in a fine-grained manner. If you categorize data with diverse information together, it will increase the training difficulty and causes model to generate low quality images).
    * In MeiweiPCB dataset, we only have one type of data (PCB surface images). Since all the defects in this dataset are visually similar, all anomalies are classified into a single class: "defect"
    * Therefore, the anomaly_types should be:
        - ['PCB', 'defect']
* Your dataset must follow the anomaly type categorization as below. For each anomaly image, it must include a paired mask image indicating where the anomaly occurred in the image.
    * Format
    ```code
    <DATASET_ROOT>/
    ├── <TEXTURE_1>/
    │   ├── anomaly_image/
    │   │   ├── <anomaly_type_1>/
    │   │   │   ├── image_1.jpg
    │   │   │   ├── image_2.jpg
    │   │   │   └── ...
    │   │   └── <anomaly_type_2>/
    │   │       ├── image_1.jpg
    │   │       ├── image_2.jpg
    │   │       └── ...
    │   └── mask/
    │       ├── <anomaly_type_1>/
    │       │   ├── image_1_mask.jpg
    │       │   ├── image_2_mask.jpg
    │       │   └── ...
    │       └── <anomaly_type_2>/
    │           ├── image_1_mask.jpg
    │           ├── image_2_mask.jpg
    │           └── ...
    └── <TEXTURE_2>/
        └── ...
    ```
    * Continue with the MeiweiPCB example (only one texure type with one anomaly type), your dataset should look like this:
    ```code
    <DATASET_ROOT>/
    └── PCB/
        ├── anomaly_image/
        │   └── defect/
        │       ├── image_1.jpg
        │       └── ...
        └── mask/
            └── defect/
                ├── image_1_mask.jpg
                └── ...
    ```    
    * The image / mask pair should have exactly same image size, and the naming is matched (mask image has suffix '_mask')

### 0.3 Preparing Pretrained Models

We will need 4 pretrained modules for training PAIDF AnomalyGen.
Make sure you have the following checkpoints downloaded and placed in the checkpoints directory with the structure below:

```shell
checkpoints/
├── nvidia/Cosmos-Predict2-2B-Text2Image/
├── nvidia/Cosmos-Predict2-14B-Text2Image/
├── google-t5/t5-large
└── NVDINOV2/
```

List of required modules:

- `nvidia/Cosmos-Predict2-2B-Text2Image`: Main text-to-world prediction backbone. 2B version.
- `nvidia/Cosmos-Predict2-14B-Text2Image`: Main text-to-world prediction backbone. 14B version.
- `google-t5/t5-large`: Google T5 model for text encoding. This is the default text encoder used by the training config (`t5_model_name`); the larger `t5-11b` (T5-XXL) is optional and only needed if you switch the config to it.
- `NVDINOV2`: NVIDIA DINOv2 visual encoder.

> The download step below fetches `t5-large` (plus the Cosmos-Predict2 backbones and the other required encoders/guardrail) by default. It does **not** download `t5-11b` — pass `--with_t5_11b` to `scripts.download_checkpoints` if your config uses T5-XXL.

#### 0.3.1 Downloading the Pretrained Cosmos-Predict2 Checkpoints

You could use the following script from Cosmos-Predict2 repo to download the required checkpoints.

This code block requires the user to set the environment variable `HF_TOKEN` to the value of your Hugging Face token which is necessary to download the checkpoints. Remember to replace `YOUR_HF_TOKEN` with your actual Hugging Face token.

<details>
<summary> <b> The equivalent setup script in the bash terminal. (Click to show) <b> </summary>

```bash
conda activate cosmos-predict2

# Login into Huggingface (You have to prepare your HF token)
HF_TOKEN="YOUR_HF_TOKEN"
huggingface-cli login --token ${HF_TOKEN} --add-to-git-credential

# Download checkpoint
CUDA_HOME=$CONDA_PREFIX python -m scripts.download_checkpoints --model_types text2image --model_sizes 2B 14B
```

</details>

In [ ]:
# Login into Huggingface (You have to prepare your HF token)
HF_TOKEN="YOUR_HF_TOKEN"
!conda run -n cosmos-predict2 hf auth login --token {HF_TOKEN} --add-to-git-credential

# Download checkpoint
!conda run -n cosmos-predict2 bash -c "CUDA_HOME=$CONDA_PREFIX python -m scripts.download_checkpoints --model_types text2image --model_sizes 2B 14B"

## Next Step

You can now proceed to the next step to train the PAIDF AnomalyGen modules: [1-training.ipynb](./1-training.ipynb)